# Style-Bert-VITS2 吹き替え自動化ツール

## 📋 準備

### 1. フォルダ構成
以下のようにファイルを配置してください：

```
Style-Bert-VITS2/
├── input_mp4/
│   ├── srt/                    # ← 全ての字幕ファイル(.srt)をここに配置
│   │   ├── 動画1.srt
│   │   ├── 動画2.srt
│   │   └── ...
│   ├── 講座A/                  # ← 動画ファイル(.mp4)はフォルダ分けしてOK
│   │   ├── 動画1.mp4
│   │   └── 動画2.mp4
│   └── 講座B/
│       └── 動画3.mp4
└── output_mp4/                 # ← 吹き替え後の動画がここに出力される
    ├── 講座A/
    │   ├── 動画1.mp4         # 吹き替え済み
    │   ├── 動画1.srt         # 字幕も一緒にコピー
    │   ├── 動画2.mp4
    │   └── 動画2.srt
    └── 講座B/
        ├── 動画3.mp4
        └── 動画3.srt
```

### 2. モデルの準備
- `model_assets/` フォルダ内に使用したいTTSモデルを配置
- 以下のコードで `model_name="モデル名"` を指定するだけでOK

### 3. 字幕ファイルの形式
- SRTファイル名は動画ファイル名と同じにしてください
  - 例: `動画1.mp4` → `動画1.srt`
- 文字コードは自動検出（UTF-8, Shift_JIS, CP932対応）
- 出力時は **UTF-8 BOM付き** で保存されます（Windowsメディアプレイヤーで文字化けしません）

## ⚙️ 機能

✅ **音声カット禁止ポリシー**
- 字幕の時間内に収まらない場合は話速を自動調整（最大2倍速まで）

✅ **英語→カタカナ自動変換**
- 字幕内の英語を日本語読みに変換（例: "Coloso" → "コロソ"）

✅ **イントロ音声重ね合わせ**
- 最初の字幕が始まるまでは元動画の音声を残します（フェードアウト付き）

✅ **重複処理スキップ**
- 既に出力済みの動画は上書きせずスキップ

## 🚀 使い方

下のセルを実行してください。

In [ ]:
from dubbing_tools import DubbingAutomation
import os

# モデル初期化（モデル名だけで自動的にファイルを検索）
dubbing = DubbingAutomation(
    model_name="ModelName",  # model_assets/モデル名/ 内のファイルを自動検索
    device="cuda",
)

# input_mp4内のすべてのMP4を自動検索
input_base = "input_mp4"
output_base = "output_mp4"
srt_folder = os.path.join(input_base, "srt")

for root, dirs, files in os.walk(input_base):
    if "srt" in root:  # srtフォルダはスキップ
        continue
        
    for file in files:
        if file.endswith('.mp4'):
            video_path = os.path.join(root, file)
            name = file.rsplit('.', 1)[0]
            srt_path = os.path.join(srt_folder, f"{name}.srt")
            
            if os.path.exists(srt_path):
                # 出力先: input_mp4/講座A → output_mp4/講座A
                relative_path = os.path.relpath(video_path, input_base)
                output_path = os.path.join(output_base, relative_path)
                
                # 既に出力済みの場合はスキップ
                if os.path.exists(output_path):
                    print(f"\nスキップ（既に存在）: {output_path}")
                    continue
                
                print(f"\n処理中: {video_path}")
                dubbing.create_dubbed_video(
                    video_path=video_path,
                    srt_path=srt_path,
                    output_path=output_path,
                    intro_only=True,  # イントロ部分のみ元の音声を重ねる
                    overlay=True,
                    audio_volume=1.0,
                    original_volume=0.3,
                )
                print(f"完了: {output_path}")